This notebook might not be usable "out of the box" and is mainly meant to give an idea how the data was calculated from the NMR bundles and the MD ensembles.

# Import modules

In [ ]:
import pandas as pd
import pickle as pkl
import barnaba as bb
from subprocess import Popen, PIPE
import regex as re
import mdtraj as md
from lib.noe_helper import *

In [2]:
%load_ext autoreload
%autoreload 2
import os
import sys
module_path = os.path.abspath(os.path.join('lib')) # or the path to your source code
sys.path.insert(0, module_path)
from lib.noe_helper import *

# Define Functions

In [3]:
def load( inp ):
    pin = open( inp, "rb" )
    return pkl.load( pin )

def save( outfile, results ):
    with open( outfile + ".pkl", "wb" ) as fp:
        pkl.dump( results, fp )

In [4]:
### HELPER FUNCTIONS TO CALCULATE NOE DISTANCES ###

# helper functions, subsititute strings. This is because the name of hydrogens is a mess
# alt = {"H2'":"1H2'","H5''":"2H5'","H5'":"1H5'","HO2'":"2HO'","H5\"":"2H5'","H5'2":"2H5'","H5'1":"1H5'"}
alt = {"H2'":"H2'1","H5''":"H5'2","H5'":"H5'1","HO2'":"HO'2"}

def sub(ss):
    at = "H" + ss.split("H")[1]
    if(at in alt):
        at = alt[at]
    return ss.split("H")[0] + at

# read experimental datafile and returns a list of labels and experimental values
def read_exp(f_exp):
    labels = []
    vals = []
    with open(f_exp) as fh:
        for line in fh:
            if("#" not in line):
                r1 = line.split()[0].split("-")[0]
                print(r1)
                r2 = line.split()[0].split("-")[1]
                print(r2)
                v1 = np.sort([r1,r2])
                print(v1)
                qq = v1[0] +"/"+ v1[1]
                
                if(qq in labels):
                    print("# DUPLICATE. Skipping data.."),
                    print(qq,vals[labels.index(qq)], line),
                else:
                    vals.append([float(line.split()[1]),float(line.split()[2])])
                    labels.append(qq)
    return labels,vals

# get labels from df column (Assignment) 
def get_labels(df, col):
    labels = []
    for asm in df.iloc[:,col]:
        r1 = asm.split()[0].split("-")[0]
        r2 = asm.split()[0].split("-")[1]
        v1 = np.sort([r1,r2])
        qq = v1[0] +"/"+ v1[1]
        labels.append(qq)
    return labels

# find indeces in topology corresponding to labels in experimental datafile
def get_idxs(labels,top):

    atoms = []
    for atom in top.atoms:
        aa = str(atom).split("-")[1]
        if(aa in alt): aa = alt[aa]
        atoms.append("%s%s" % (str(atom).split("-")[0],aa))
    pairs = []
    for el in labels:
        ss  = el.split("/")
        at1 = sub(ss[0])
        at2 = sub(ss[1])
        if(at1 in atoms and at2 in atoms):
            pairs.append([atoms.index(at1),atoms.index(at2)])
        else:
            print("# Warning: Either %s or %s are missing" % (at1,at2))
            return 0
    if len(pairs) != len(labels):
        print("# Found only %d pairs out of %d" % (len(pairs),len(labels)))
    return np.array(pairs)

def group_by_heading( some_source ):
    buffer= []
    for line in some_source:
        if line.startswith( " ASSI" ):
            if buffer: yield buffer
            buffer= [ line.strip().strip(')') ]
        else:
            buffer.append( line.strip().strip(')') )
    yield buffer

# From NMR bundles

Back-calculate experimental observables from

Since some of the forward models need certain PDB formats we have to reformat the PDB files.

In [5]:
# NMR bundle directory:
bundles_dir = 'nmrbundle_data'

# Experimental Results directory
exp_data_dir = "exp_data"

# Bundle identifiers:
bundles = ['1_3', '2_2', 'Farfar1', 'Farfar2', 'alphafold3', "1ZIH"]


## RDCs

We use the pf1-phage prediction method implemented in *PALES** to back-calculate RDCs. In order to do that we use the script ```calc_rdc_bundles.py``` which submits each frame to *PALES*, predicts the alignment tensor and calculates RDCs, parses the output and saves the D values in a Pickle file.

The actual command line to run *PALES is*: ```pales-linux -inD exp_data/exp_rdc_pales.tab -pdb {pdb_tmp} -outD {outd_tmp} -pf1 -H -wv 0.05```


*Zweckstetter, M. NMR: Prediction of molecular alignment from structure using the PALES software. Nat. Protoc. 3, 679–690 (2008).

In [7]:
 # RDCs (works)
rdc_exp_input = pd.read_csv(f'{exp_data_dir}/exp_rdc_pales.tab', names=['RESID_I', 'RESNAME_I', 'ATOMNAME_I', 'RESID_J', 'RESNAME_J',
       'ATOMNAME_J', 'D', 'DD', 'W'], delim_whitespace=True, skiprows=5)
# all residues:
rdc_exp_labels_all = [f"{row[1]['RESNAME_I'][0]}{row[1]['RESID_I']}_{row[1]['ATOMNAME_I']}-{row[1]['ATOMNAME_J']}" for row in rdc_exp_input.iterrows()]
# loop residues:
rdc_exp_labels_loop = [f"{row[1]['RESNAME_I'][0]}{row[1]['RESID_I']}_{row[1]['ATOMNAME_I']}-{row[1]['ATOMNAME_J']}" for row in rdc_exp_input.iterrows() if row[1]['RESID_I'] in [5,6,7,8,9,10]]

In [73]:
for bundle in bundles:
    process = Popen(f'python {bundles_dir}/calc_rdc_bundles.py {bundle}', shell=True, stdin=PIPE, stdout=PIPE, universal_newlines=True)
    process.wait() # needed to run one process at a time (because of using temporary files etc.)

Parse the output and write BME readable files:

In [8]:
for bundle in bundles:
    d_df = load(f'{bundles_dir}/nmrbundle_{bundle}_d_df_pf1.pkl')
    for i in rdc_exp_labels_all: # missing residues in 1RNGaxis=1
        if i not in d_df.columns:
            d_df[i] = np.full((len(d_df)), np.nan)
    
    # All res:
    d_df[rdc_exp_labels_all].to_csv(f'{bundles_dir}/nmrbundle_{bundle}_rdc_all_bme.dat', header=False, sep='\t', na_rep=np.nan)
    # Loop only:
    d_df[rdc_exp_labels_loop].to_csv(f'{bundles_dir}/nmrbundle_{bundle}_rdc_loop_bme.dat', header=False, sep='\t', na_rep=np.nan)

The predicted RDCs need to be scaled to the experimental values before comparing them. To reduce the effect of over-fitting we scale the RDCs based on all measurements and since we treat NMR structures as a bundle of structures rather than an ensemble we do the scaling for all individual structures.

In [ ]:
for bundle in bundles:
    rdc_bundle = np.loadtxt(f'{bundles_dir}/nmrbundle_{bundle}_rdc_all_bme.dat')
    L = np.zeros(len(rdc_bundle))
    for i in range(len(L)):
        masked = np.ma.array(rdc_bundle[i,1:], mask=np.isnan(rdc_bundle[i,1:])) # mask NaNs 
        L[i] = np.sum(np.array(rdc_exp_input['D'])*masked)/np.sum(masked*masked)
    np.save(f'{bundles_dir}/nmrbundle_{bundle}_rdc_all_bme_L', L)
    rdc_bundle = np.loadtxt(f'{bundles_dir}/nmrbundle_{bundle}_rdc_all_bme.dat')
    L = np.zeros(len(rdc_bundle))
    for i in range(len(L)):
        masked = np.ma.array(rdc_bundle[i,1:], mask=np.isnan(rdc_bundle[i,1:])) # mask NaNs 
        L[i] = np.sum(np.array(rdc_exp_input['D'])*masked)/np.sum(masked*masked)
    np.save(f'{bundles_dir}/nmrbundle_{bundle}_rdc_loop_bme_L', L)

## $^3$J-couplings

We use Barnaba* to calculate the $^3$J scalar couplings 2H5H4, H1H2, H2H3, H3P, C4Pb, 1H5P, 1H5H4, C4Pe, 2H5P and H3H4.

Definition: $A cos^2 (\theta + \phi) + B cos (\theta + \phi) + C$

*Bottaro, S. et al. Barnaba: software for analysis of nucleic acid structures and trajectories. Rna 25, 219–231 (2019)

In [77]:
# A (Hz), B (Hz), C (Hz), _, phi (rad)
bb.definitions.couplings_karplus

{'H1H2': [9.67, -2.03, 0.0, 0.0, 0.0],
 'H2H3': [9.67, -2.03, 0.0, 0.0, 0.0],
 'H3H4': [9.67, -2.03, 0.0, 0.0, 0.0],
 '1H5P': [15.3, -6.1, 1.6, 0.0, -2.094395],
 '2H5P': [15.3, -6.1, 1.6, 0.0, 2.094395],
 'C4Pb': [6.9, -3.4, 0.7, 0.0, 0.0],
 '1H5H4': [9.7, -1.8, 0.0, 0.0, -2.094395],
 '2H5H4': [9.7, -1.8, 0.0, 0.0, 0.0],
 'H3P': [15.3, -6.1, 1.6, 0.0, 2.094395],
 'C4Pe': [6.9, -3.4, 0.7, 0.0, 0.0],
 'H1C2/4': [4.7, 2.3, 0.1, 0.0, -1.0471975],
 'H1C6/8': [4.5, -0.6, 0.1, 0.0, -1.0471975]}

In [78]:
j3_exp_labels = list(pd.read_csv('exp_data/exp_j3_bme.dat', delim_whitespace=True).index) # ONLY EXTENDED-LOOP RESIDUES!!!
j3_exp_couplings = list(set([ i.split('-')[-1] for i in j3_exp_labels ]))

# Never used?
barnaba_to_exp = {'H1H2':"H1',H2'", 'H2H3':"H2',H3'", 'H3H4':"H3',H4'",\
                '1H5P':"H5'i,Pi", '2H5P':"H5''i,Pi",\
                'C4Pb':"C4'i,Pi", '1H5H4':"NA", '2H5H4':"NA", \
                'H3P':"H3'i,Pi+1" ,'C4Pe':"C4'i,Pi+1", \
                'H1C2/4':"NA", 'H1C6/8':"NA"}

# the shape of couplings is (nframes, nresidues, ncouplings)
# only look at bundle A
for bundle in bundles:
    traj_file   = f'{bundles_dir}/Bundle_{bundle}.pdb'
    top_file    = f'{bundles_dir}/Bundle_{bundle}.pdb'

    couplings,residues = bb.jcouplings(traj_file, topology=top_file, couplings=j3_exp_couplings)
    print( f'Calculated jcouplings for {j3_exp_couplings}.' )
    print( couplings.shape )

    couplings_avg = np.average( couplings, 0 )
    print(couplings_avg.shape)
    
    j3_calc_ = pd.DataFrame()
    for i,ii in enumerate([ i.split('_')[0]+i.split('_')[1] for i in residues]):
        for j,jj in enumerate(j3_exp_couplings):
            print(f'Bundle {traj_file} Coupling: {ii}-{jj}')
            j3_calc_[f'{ii}-{jj}'] = couplings[:,i,j]

    ll = []
    for i in j3_exp_labels:
        ll.append(j3_calc_[i])
    pd.DataFrame(ll).T.to_csv(f'{bundles_dir}/nmrbundle_{bundle}_j3_loop_bme.dat', sep=' ', header=False)

# Loading nmrbundle_data/Bundle_1_3.pdb 
# Skipping unknown residue URI11 


Calculated jcouplings for ['2H5P', 'C4Pe', '1H5P', 'H3P', 'H1H2', 'C4Pb', 'H3H4'].
(20, 13, 7)
(13, 7)
Bundle nmrbundle_data/Bundle_1_3.pdb Coupling: G1-2H5P
Bundle nmrbundle_data/Bundle_1_3.pdb Coupling: G1-C4Pe
Bundle nmrbundle_data/Bundle_1_3.pdb Coupling: G1-1H5P
Bundle nmrbundle_data/Bundle_1_3.pdb Coupling: G1-H3P
Bundle nmrbundle_data/Bundle_1_3.pdb Coupling: G1-H1H2
Bundle nmrbundle_data/Bundle_1_3.pdb Coupling: G1-C4Pb
Bundle nmrbundle_data/Bundle_1_3.pdb Coupling: G1-H3H4
Bundle nmrbundle_data/Bundle_1_3.pdb Coupling: G2-2H5P
Bundle nmrbundle_data/Bundle_1_3.pdb Coupling: G2-C4Pe
Bundle nmrbundle_data/Bundle_1_3.pdb Coupling: G2-1H5P
Bundle nmrbundle_data/Bundle_1_3.pdb Coupling: G2-H3P
Bundle nmrbundle_data/Bundle_1_3.pdb Coupling: G2-H1H2
Bundle nmrbundle_data/Bundle_1_3.pdb Coupling: G2-C4Pb
Bundle nmrbundle_data/Bundle_1_3.pdb Coupling: G2-H3H4
Bundle nmrbundle_data/Bundle_1_3.pdb Coupling: C3-2H5P
Bundle nmrbundle_data/Bundle_1_3.pdb Coupling: C3-C4Pe
Bundle nmrbundle_da

# Loading nmrbundle_data/Bundle_2_2.pdb 
# Skipping unknown residue URI11 
# Loading nmrbundle_data/Bundle_Farfar1.pdb 


Calculated jcouplings for ['2H5P', 'C4Pe', '1H5P', 'H3P', 'H1H2', 'C4Pb', 'H3H4'].
(20, 13, 7)
(13, 7)
Bundle nmrbundle_data/Bundle_2_2.pdb Coupling: G1-2H5P
Bundle nmrbundle_data/Bundle_2_2.pdb Coupling: G1-C4Pe
Bundle nmrbundle_data/Bundle_2_2.pdb Coupling: G1-1H5P
Bundle nmrbundle_data/Bundle_2_2.pdb Coupling: G1-H3P
Bundle nmrbundle_data/Bundle_2_2.pdb Coupling: G1-H1H2
Bundle nmrbundle_data/Bundle_2_2.pdb Coupling: G1-C4Pb
Bundle nmrbundle_data/Bundle_2_2.pdb Coupling: G1-H3H4
Bundle nmrbundle_data/Bundle_2_2.pdb Coupling: G2-2H5P
Bundle nmrbundle_data/Bundle_2_2.pdb Coupling: G2-C4Pe
Bundle nmrbundle_data/Bundle_2_2.pdb Coupling: G2-1H5P
Bundle nmrbundle_data/Bundle_2_2.pdb Coupling: G2-H3P
Bundle nmrbundle_data/Bundle_2_2.pdb Coupling: G2-H1H2
Bundle nmrbundle_data/Bundle_2_2.pdb Coupling: G2-C4Pb
Bundle nmrbundle_data/Bundle_2_2.pdb Coupling: G2-H3H4
Bundle nmrbundle_data/Bundle_2_2.pdb Coupling: C3-2H5P
Bundle nmrbundle_data/Bundle_2_2.pdb Coupling: C3-C4Pe
Bundle nmrbundle_da

# Loading nmrbundle_data/Bundle_Farfar2.pdb 
# Loading nmrbundle_data/Bundle_alphafold3.pdb 
c:\Users\david_leopold\anaconda3\Lib\site-packages\mdtraj\formats\pdb\pdbfile.py:200: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn('Unlikely unit cell vectors detected in PDB file likely '


Calculated jcouplings for ['2H5P', 'C4Pe', '1H5P', 'H3P', 'H1H2', 'C4Pb', 'H3H4'].
(10, 14, 7)
(14, 7)
Bundle nmrbundle_data/Bundle_Farfar2.pdb Coupling: G1-2H5P
Bundle nmrbundle_data/Bundle_Farfar2.pdb Coupling: G1-C4Pe
Bundle nmrbundle_data/Bundle_Farfar2.pdb Coupling: G1-1H5P
Bundle nmrbundle_data/Bundle_Farfar2.pdb Coupling: G1-H3P
Bundle nmrbundle_data/Bundle_Farfar2.pdb Coupling: G1-H1H2
Bundle nmrbundle_data/Bundle_Farfar2.pdb Coupling: G1-C4Pb
Bundle nmrbundle_data/Bundle_Farfar2.pdb Coupling: G1-H3H4
Bundle nmrbundle_data/Bundle_Farfar2.pdb Coupling: G2-2H5P
Bundle nmrbundle_data/Bundle_Farfar2.pdb Coupling: G2-C4Pe
Bundle nmrbundle_data/Bundle_Farfar2.pdb Coupling: G2-1H5P
Bundle nmrbundle_data/Bundle_Farfar2.pdb Coupling: G2-H3P
Bundle nmrbundle_data/Bundle_Farfar2.pdb Coupling: G2-H1H2
Bundle nmrbundle_data/Bundle_Farfar2.pdb Coupling: G2-C4Pb
Bundle nmrbundle_data/Bundle_Farfar2.pdb Coupling: G2-H3H4
Bundle nmrbundle_data/Bundle_Farfar2.pdb Coupling: C3-2H5P
Bundle nmrbund

# Loading nmrbundle_data/Bundle_1ZIH.pdb 


## CCRs

We use the script ```calc_ccr.py``` to back-calculate CCRs from pdb files or trajectories
Important notes:
- The PDB naming has to be in a specific format (check the script if in doubt).
- Here, $\Gamma$-HCP (C4p-P, C3p-P-plus, C4p-P-plus) and $\Gamma$-HCNCH (C1-CC) are measured at 600 MHz (14.09 T) while $\Gamma$-HCCH (C1-C2, C3-C4) is measured at 700 MHz (16.44 T) which means that in theory the script has to be executed twice and B0 has to be adjusted accordingly. However, $\Gamma$-HCCH couplings are not influenced by B0 which is why we can ignore this for now.

Calculate CCRs from bundles:

Note: certain atoms need specific naming in the .pdb files. Change manualy if necessary 

In [ ]:
# make sure the pdbs use OP1 and OP2 instead of O1P and O2P 

ccrs = ["HCCH", "HCN", "HCP"]
#ccrs = ["HCN"]
field = 16.44 #16.44 T for 700 MHz

for ccr in ccrs:
    ccr_exp_labels_loop  = list(pd.read_csv(f'exp_data/exp_ccr_{ccr}.dat', delim_whitespace=True, usecols=[0]).index)
    ccr_exp_loop  = np.loadtxt(f'exp_data/exp_ccr_{ccr}.dat', usecols=[1,2])

    # Check the script and change output name etc.
    for bundle in bundles:
        process = Popen(f'python {bundles_dir}/calc_ccr_2.py {bundles_dir}/Bundle_{bundle}_ccr.pdb {field}', shell=True, stdin=PIPE, stdout=PIPE, universal_newlines=True)
        process.wait() # needed to run one process at a time (because of using temporary files etc.)
    
    for bundle in bundles:
        tmp_ccr = pd.read_csv(f'{bundles_dir}/Bundle_{bundle}_ccr_calc.dat', delim_whitespace=True, index_col=0)
        tmp_ccr_loop = tmp_ccr[ccr_exp_labels_loop]
        # Check if all data points are contained in the bundle:
        print(list(tmp_ccr_loop.columns))
        # Combine 700 MHz data for gamma-HCCH (C1-C2, C3-C4) couplings with the rest at 600 MHz:
        tmp_ccr_loop.to_csv(f'{bundles_dir}/nmrbundle_{bundle}_ccr_calc_{ccr}_loop_bme.dat', header=False, sep=' ')

['C5:C1-C2', 'G6:C1-C2', 'C7:C1-C2', 'A8:C1-C2', 'G10:C1-C2', 'G6:C3-C4', 'C7:C3-C4', 'A9:C3-C4', 'G10:C3-C4']
['C5:C1-C2', 'G6:C1-C2', 'C7:C1-C2', 'A8:C1-C2', 'G10:C1-C2', 'G6:C3-C4', 'C7:C3-C4', 'A9:C3-C4', 'G10:C3-C4']
['C5:C1-C2', 'G6:C1-C2', 'C7:C1-C2', 'A8:C1-C2', 'G10:C1-C2', 'G6:C3-C4', 'C7:C3-C4', 'A9:C3-C4', 'G10:C3-C4']
['C5:C1-C2', 'G6:C1-C2', 'C7:C1-C2', 'A8:C1-C2', 'G10:C1-C2', 'G6:C3-C4', 'C7:C3-C4', 'A9:C3-C4', 'G10:C3-C4']
['C5:C1-C2', 'G6:C1-C2', 'C7:C1-C2', 'A8:C1-C2', 'G10:C1-C2', 'G6:C3-C4', 'C7:C3-C4', 'A9:C3-C4', 'G10:C3-C4']
['C5:C1-C2', 'G6:C1-C2', 'C7:C1-C2', 'A8:C1-C2', 'G10:C1-C2', 'G6:C3-C4', 'C7:C3-C4', 'A9:C3-C4', 'G10:C3-C4']
['C5:C1-CC', 'C7:C1-CC', 'A8:C1-CC']
['C5:C1-CC', 'C7:C1-CC', 'A8:C1-CC']
['C5:C1-CC', 'C7:C1-CC', 'A8:C1-CC']
['C5:C1-CC', 'C7:C1-CC', 'A8:C1-CC']
['C5:C1-CC', 'C7:C1-CC', 'A8:C1-CC']
['C5:C1-CC', 'C7:C1-CC', 'A8:C1-CC']
['C5:C4p-P', 'G6:C4p-P', 'C7:C4p-P', 'A8:C4p-P', 'A9:C4p-P', 'C5:C3p-P-plus', 'A9:C3p-P-plus', 'C5:C4p-P-plus', 

NOTE: Don't forget about adding nans for missing measurements in 1RNG when compiling a data file for all residues (including stem).

## NOEs

We load NOEs from an ARIA output file after they have been converted to distances (\AA). For this, we use the NOEs from bundle F because those have been restraint the most by other types of data. Generally, they don't differ significantly between the different bundles.

Load and parse experimental NOE distance file:

In [6]:
seq = {1:'G', 2:'G', 3:'C', 4:'A', 5:'C', 6:'G', 7:'C', 8:'A', 9:'A', 10:'G', 11:'U', 12:'G', 13:'C', 14:'C'}
for bundle in bundles:    
    df_noe_exp = pd.DataFrame()
    
    with open(f'exp_data/exp_noe_2_2.tbl') as f: # use NOEs from bundle 2+2
        
        for heading_and_lines in group_by_heading( f ):
            heading= heading_and_lines[0]
            lines= heading_and_lines[1:]
            
            resid1 = int(re.search('resid (\d+)', lines[0] ).group(1))
            name1 = re.search('name (.+)', lines[0] ).group(1).strip()
            resid2 = int(re.search('resid (\d+)', lines[1] ).group(1))
            name2 = re.search('name (.+)', lines[1] ).group(1).strip()
            
            row = {'Assignment': f'{seq[resid1]}{resid1}{name1}-{seq[resid2]}{resid2}{name2}', 'distance':float(lines[2].split()[0]), 'upper_err':float(lines[2].split()[1]), 'lower_err':float(lines[2].split()[2])}
            df_row = pd.DataFrame.from_dict(row, orient='index')
            df_noe_exp = pd.concat([df_noe_exp, df_row], axis=1)
        df_noe_exp = df_noe_exp.T

Make separate dfs for loop NOEs:

In [ ]:
residues = ['G1', 'G2', 'C3', 'A4', 'C5', 'G6', 'C7', 'A8', 'A9', 'G10', 'U11', 'G12', 'C13', 'C14']
loop_c5 = ['C5','G6', 'C7', 'A8', 'A9','G10']


def create_df_noe_exp_loop( loop):
    a_loop = []
    for i,n in df_noe_exp[:].iterrows():
        a = n['Assignment'].split('-')
        r1 = re.split('(^[A-Z]\d+)', a[0])[1]
        r2 = re.split('(^[A-Z]\d+)', a[1])[1]

        if r1 in loop and r2 in loop:
            a_loop.append(n['Assignment'])
            print(f"Added Assignment: {n['Assignment']}")

    print(f'Loop NOEs: {len(a_loop)}')
    # print( f'Loop NOEs: {len(a_loop)}\nStem NOEs: {len(a_stem)}\nIntersect: {len(a_both)}' )
    return df_noe_exp.loc[df_noe_exp.Assignment.isin(a_loop)], a_loop

df_noe_exp_loop, a_loop = create_df_noe_exp_loop(loop_c5)
df_noe_exp_all, all = create_df_noe_exp_loop(residues)

print("shape",df_noe_exp_loop.shape)
print("shape",df_noe_exp_all.shape)

Added Assignment: C5H2'-C5H1'
Added Assignment: C5H2'-G6H1'
Added Assignment: C5H3'-C5H1'
Added Assignment: C5H5-G6H8
Added Assignment: C7H2'-C7H1'
Added Assignment: C7H2'-C7H5
Added Assignment: A8H1'-A9H2
Added Assignment: A8H3'-A8H1'
Added Assignment: A8H5'2-A8H5'1
Added Assignment: G10H5'2-G10H5'1
Added Assignment: C5H1'-C5H6
Added Assignment: C5H2'-C5H6
Added Assignment: C5H3'-C5H6
Added Assignment: C5H5-C5H6
Added Assignment: C5H5'2-C5H6
Added Assignment: G6H2'-G6H1'
Added Assignment: G6H3'-G6H1'
Added Assignment: G6H3'-G6H8
Added Assignment: G6H5'2-G6H5'1
Added Assignment: C7H1'-C7H6
Added Assignment: C7H2'-C7H6
Added Assignment: C7H5'2-C7H4'
Added Assignment: C7H5'2-A8H8
Added Assignment: A9H1'-A9H2
Added Assignment: A9H4'-A9H1'
Added Assignment: G10H2'-G10H8
Added Assignment: G6H2'-G6H8
Added Assignment: G6H2'-A8H8
Added Assignment: G6H4'-G6H1'
Added Assignment: G6H5'1-C7H5
Added Assignment: C7H2'-A8H8
Added Assignment: C7H3'-C7H1'
Added Assignment: C7H3'-C7H6
Added Assignment:

Write BME readable file for the experimental loop noes:

In [22]:
df_noe_exp_loop_tobme = df_noe_exp_loop.copy().drop(columns=['lower_err', 'upper_err'])
df_noe_exp_loop_tobme['avg_err'] = ((df_noe_exp_loop['lower_err']+df_noe_exp_loop['upper_err'])/2)

df_noe_exp_all_tobme = df_noe_exp_all.copy().drop(columns=['lower_err', 'upper_err'])
df_noe_exp_all_tobme['lower_err'] = (df_noe_exp_all['lower_err'])
df_noe_exp_all_tobme['upper_err'] = (df_noe_exp_all['upper_err'])


In [23]:
with open(f'exp_data/exp_noe_loop_bme.dat', 'w') as f:
    f.write('# DATA=NOE POWER=6\n')
    df_noe_exp_loop_tobme.to_csv(f, index= False, header = False, sep = '\t', lineterminator='\n', float_format='%.2f') #df_noe_exp_tobme[df_noe_exp_tobme['Assignment'].isin(a_loop)]

with open(f'exp_data/exp_noe_all_SI.dat', 'w') as f:
    f.write('# DATA=NOE POWER=6\n')
    df_noe_exp_all_tobme.to_csv(f, index= False, header = False, sep = '\t', lineterminator='\n', float_format='%.2f') 



In [ ]:
# Calc distances from the bundles (1ZIH has to be one separately due to missing/different residues)
for bundle in bundles: #all except last 1ZIH
    print(bundle)
    traj_ = md.load_pdb(f'nmrbundle_data/Bundle_{bundle}.pdb')
    # labels = get_labels(df_noe_exp, 0) # For all NOEs
    labels = get_labels(df_noe_exp_loop, 0)
    pairs = get_idxs( labels, traj_.topology )
    # calculate distances multiply by 10 to convert to angs
    dists = 10.0*md.compute_distances(traj_,pairs)
    df_noe = pd.DataFrame(dists)
    
    df_noe.columns = list(df_noe_exp_loop.Assignment)
    # Save to BME format:
    df_noe[a_loop].to_csv(f'nmrbundle_data/nmrbundle_{bundle}_dists_loop.dat', sep='\t', header=False)

1_3
2_2
Farfar1
Farfar2
alphafold3
1ZIH


c:\Users\david_leopold\anaconda3\Lib\site-packages\mdtraj\formats\pdb\pdbfile.py:200: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn('Unlikely unit cell vectors detected in PDB file likely '


# From MD ensembles

## RDCs

We use the pf1-phage prediction method implemented in *PALES** to back-calculate RDCs. In order to do that we use the script ```calc_rdc_traj.py``` which submits each frame to *PALES*, predicts the alignment tensor and calculates RDCs, parses the output and saves the D values in a Pickle file.
Just run the code block! Script is now imported.

The actual command line to run *PALES is*: ```pales-linux -inD exp_data/exp_rdc_pales.tab -pdb {pdb_tmp} -outD {outd_tmp} -pf1 -H -wv 0.05```


*Zweckstetter, M. NMR: Prediction of molecular alignment from structure using the PALES software. Nat. Protoc. 3, 679–690 (2008).

In [ ]:
# After running the script above (calc_rdc_traj_.py):
df = load(f'gcaa_simulations/d_df_pf1.pkl') # Change

# All to BME
df.T.to_csv(f'calc_data/calc_rdc_bme.dat', header=False, sep='\t')
# Loop to BME:
df.T[rdc_exp_labels_loop].to_csv(f'calc_data/calc_rdc_loop_bme.dat', header=False, sep='\t')

## $^3$J-couplings

We use Barnaba* to calculate the $^3$J scalar couplings 2H5H4, H1H2, H2H3, H3P, C4Pb, 1H5P, 1H5H4, C4Pe, 2H5P and H3H4. (From simulations)

Definition: $A cos^2 (\theta + \phi) + B cos (\theta + \phi) + C$

*Bottaro, S. et al. Barnaba: software for analysis of nucleic acid structures and trajectories. Rna 25, 219–231 (2019)

In [ ]:
# 3J couplings
j3_exp_labels = list(pd.read_csv('exp_data/exp_j3_bme.dat', delim_whitespace=True).index) # ONLY EXTENDED-LOOP RESIDUES!!!
j3_exp_couplings = list(set([ i.split('-')[-1] for i in j3_exp_labels ]))
###

#Never used?
barnaba_to_exp = {'H1H2':"H1',H2'", 'H2H3':"H2',H3'", 'H3H4':"H3',H4'",                                                    
                '1H5P':"H5'i,Pi", '2H5P':"H5''i,Pi",                                                        
                'C4Pb':"C4'i,Pi", '1H5H4':"NA", '2H5H4':"NA",                                                            
                'H3P':"H3'i,Pi+1" ,'C4Pe':"C4'i,Pi+1",                                                                   
                'H1C2/4':"NA", 'H1C6/8':"NA"}                                                                               
###
###
# the shape of couplings is (nframes, nresidues, ncouplings)
###
traj_file   = f'gcaa_simulations/concat_traj_nopbc.xtc'
top_file    = f'gcaa_simulations/initial_nopbc_mdtraj_ccr.pdb'
###
couplings,residues = bb.jcouplings(traj_file, topology=top_file, couplings=j3_exp_couplings)
print( f'Calculated jcouplings for {j3_exp_couplings}.' )
print( couplings.shape )
###
couplings_avg = np.average( couplings, 0 )
print(couplings_avg.shape)
###
j3_calc_ = pd.DataFrame()
for i,ii in enumerate([ i.split('_')[0]+i.split('_')[1] for i in residues]):
    for j,jj in enumerate(j3_exp_couplings):
        j3_calc_[f'{ii}-{jj}'] = couplings[:,i,j]
###
ll = []
for i in j3_exp_labels:
    #print("label", i,j3_calc_[i])
    ll.append(j3_calc_[i])
pd.DataFrame(ll).T.to_csv(f'calc_data/calc_j3_bme.dat', sep=' ', header=False)

# Loading gcaa_simulations/concat_traj_nopbc.xtc 


Calculated jcouplings for ['H1H2', '2H5P', 'C4Pe', '1H5P', 'C4Pb', 'H3P', 'H3H4'].
(20100, 14, 7)
(14, 7)
label C5-H1H2 0        0.772062
1        0.486416
2        2.817808
3        0.378287
4        0.109403
           ...   
20095    0.675581
20096    0.705017
20097    0.983659
20098    0.046948
20099    2.032618
Name: C5-H1H2, Length: 20100, dtype: float64
label G6-H1H2 0        0.540178
1        2.402632
2        0.876766
3        0.065437
4        0.181589
           ...   
20095    0.117073
20096    0.386599
20097    0.202147
20098    0.247499
20099    0.399903
Name: G6-H1H2, Length: 20100, dtype: float64
label C7-H1H2 0        10.853304
1         9.131660
2         0.156973
3        -0.070429
4         0.988500
           ...    
20095    10.583082
20096    10.448660
20097    10.249973
20098    11.198127
20099    11.513083
Name: C7-H1H2, Length: 20100, dtype: float64
label A8-H1H2 0       -0.082368
1        0.237606
2        0.039159
3       -0.078808
4       -0.044633
        

## CCRs

We use the script ```calc_ccr.py``` to back-calculate CCRs from pdb files or trajectories
Important notes:
- The PDB naming has to be in a specific format (check the script if in doubt).
- Here, $\Gamma$-HCP (C4p-P, C3p-P-plus, C4p-P-plus) and $\Gamma$-HCNCH (C1-CC) are measured at 600 MHz (14.09 T) while $\Gamma$-HCCH (C1-C2, C3-C4) is measured at 700 MHz (16.44 T) which means that in theory the script has to be executed twice and B0 has to be adjusted accordingly. However, $\Gamma$-HCCH couplings are not influenced by B0 which is why we can ignore this for now.

In [ ]:
ccrs = ["HCCH", "HCN", "HCP"]
for ccr in ccrs:
    ccr_exp_labels_loop  = list(pd.read_csv(f'exp_data/exp_ccr_{ccr}.dat', delim_whitespace=True, usecols=[0]).index)
    # ccr_exp_loop  = np.loadtxt('exp_data/exp_ccr.dat', usecols=[1,2]) why is this here?

    # Check the script and change output name etc.
    folder_ccr = "calc_data"
    traj_file   = f'gcaa_simulations/concat_traj_nopbc.xtc'             # due to the nature of calc_ccr.py and its application for nmr_bundles, the output file will be not in the calc_Data folder and needs renaming 
    top_file    = f'gcaa_simulations/initial_nopbc_mdtraj_ccr.pdb'      # due to the nature of calc_ccr.py and its application for nmr_bundles, the output file will be not in the calc_Data folder and needs renaming 

    from importlib import reload

    import nmrbundle_data.calc_ccr as ccr_calc
    # Reload if changes were made on the fly
    reload(ccr_calc)

    traj = md.load(traj_file, top=top_file)
    field = 16.44 #16.44 T for 700 MHz

    atom_renaming = {
                    "H5'2":"2H5'",
                    "H5'1":"1H5'",
                    # "H2'1":"H2'",
                    "H2'":"1H2'",

                        }


    tmp_ccr = ccr_calc.calc(traj, atom_renaming)
    tmp_ccr_loop = tmp_ccr[ccr_exp_labels_loop]
    # Check if all data points are contained in the bundle:
    print(list(tmp_ccr_loop.columns))
    # Save as BME readable file:
    tmp_ccr_loop.to_csv(f'{folder_ccr}/calc_ccr_{ccr}_loop_bme.dat', header=False, sep=' ', float_format='%8.4e')

renaming G1-H5'1 to G1-1H5'
renaming G1-H5'2 to G1-2H5'
renaming G1-H2' to G1-1H2'
renaming G2-H5'1 to G2-1H5'
renaming G2-H5'2 to G2-2H5'
renaming G2-H2' to G2-1H2'
renaming C3-H5'1 to C3-1H5'
renaming C3-H5'2 to C3-2H5'
renaming C3-H2' to C3-1H2'
renaming A4-H5'1 to A4-1H5'
renaming A4-H5'2 to A4-2H5'
renaming A4-H2' to A4-1H2'
renaming C5-H5'1 to C5-1H5'
renaming C5-H5'2 to C5-2H5'
renaming C5-H2' to C5-1H2'
renaming G6-H5'1 to G6-1H5'
renaming G6-H5'2 to G6-2H5'
renaming G6-H2' to G6-1H2'
renaming C7-H5'1 to C7-1H5'
renaming C7-H5'2 to C7-2H5'
renaming C7-H2' to C7-1H2'
renaming A8-H5'1 to A8-1H5'
renaming A8-H5'2 to A8-2H5'
renaming A8-H2' to A8-1H2'
renaming A9-H5'1 to A9-1H5'
renaming A9-H5'2 to A9-2H5'
renaming A9-H2' to A9-1H2'
renaming G10-H5'1 to G10-1H5'
renaming G10-H5'2 to G10-2H5'
renaming G10-H2' to G10-1H2'
renaming U11-H5'1 to U11-1H5'
renaming U11-H5'2 to U11-2H5'
renaming U11-H2' to U11-1H2'
renaming G12-H5'1 to G12-1H5'
renaming G12-H5'2 to G12-2H5'
renaming G12-H2

## NOEs

In [ ]:
dfs_noe = {}
#print(df_noe_exp_loop)
labels = get_labels(df_noe_exp_loop, 0)
print(labels)

traj_file   = f'gcaa_simulations/concat_traj_nopbc.xtc'
top_file    = f'gcaa_simulations/initial_nopbc_mdtraj_ccr.pdb'

traj  = md.load( traj_file, top = top_file ) # load traj
pairs = get_idxs( labels, traj.topology )

print(traj)
print(pairs)
# calculate distances multiply by 10 to convert to angs
dists = 10.0*md.compute_distances(traj,pairs)
df_noe = pd.DataFrame(dists)

df_noe.columns = list(df_noe_exp_loop['Assignment'])
print(df_noe.shape)

df_noe.to_csv(f'calc_data/calc_noe_loop_bme.dat', sep='\t', header=False, float_format='%.2f')

["C5H1'/C5H2'", "C5H2'/G6H1'", "C5H1'/C5H3'", 'C5H5/G6H8', "C7H1'/C7H2'", "C7H2'/C7H5", "A8H1'/A9H2", "A8H1'/A8H3'", "A8H5'1/A8H5'2", "G10H5'1/G10H5'2", "C5H1'/C5H6", "C5H2'/C5H6", "C5H3'/C5H6", 'C5H5/C5H6', "C5H5'2/C5H6", "G6H1'/G6H2'", "G6H1'/G6H3'", "G6H3'/G6H8", "G6H5'1/G6H5'2", "C7H1'/C7H6", "C7H2'/C7H6", "C7H4'/C7H5'2", "A8H8/C7H5'2", "A9H1'/A9H2", "A9H1'/A9H4'", "G10H2'/G10H8", "G6H2'/G6H8", "A8H8/G6H2'", "G6H1'/G6H4'", "C7H5/G6H5'1", "A8H8/C7H2'", "C7H1'/C7H3'", "C7H3'/C7H6", "A8H8/C7H3'", "C7H1'/C7H4'", "C7H3'/C7H4'", "C7H5'1/C7H6", "A8H1'/A8H2", "A8H3'/A8H8", "A8H1'/A8H4'", "A8H4'/A8H5'1", "A9H5'1/A9H5'2", 'C5H41/C5H42', 'C5H42/G10H1', 'C5H41/C5H5', "C5H5'1/C5H6", 'C5H6/G6H8', "G6H1'/G6H8", "G10H1/G6H1'", "G6H5'2/G6H8", "A8H8/C7H4'", 'C7H5/C7H6', "A8H1'/A9H8", "A8H2'/A8H8", "A8H5'1/A8H8", "A8H5'2/A8H8", "A9H3'/G10H8", "G10H5'2/G10H8", "C5H2'/G6H8", "C5H3'/G6H8", 'C5H41/G10H1', 'C5H42/C5H5', "C7H3'/C7H5'2", "C7H5'1/C7H5'2", "C7H5'2/C7H6", "A8H1'/A8H8", "A8H1'/A8H2'", "A8H2'/A9